<a href="https://colab.research.google.com/github/tanjun8802/Mase_EDGE/blob/jtan%2Fdev/EDGE_NAS_Search.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Mase_EDGE - Optuna Study and Executorch export Pipeline

Mase EDGE performs Optuna study for a specific Android phone that connects to the host PC. Alongside optimisation, the Optuna study will gather the phone information obtained through adb.exe (Android Developer mode) and define them as metrics to be studied. For example, based on the available hardware resources on the phone, alter the delegation ratio between the hardware for optimal performance development.

In [ ]:
# For colab use

# !git clone --branch jtan/dev https://github.com/tanjun8802/Mase_EDGE.git 
# %cd /content
# !rm -rf Mase_EDGE/mase
# %cd Mase_EDGE
!git submodule update --init --recursive
%cd mase/
!pip install -e .

In [ ]:
# Get the dataset (Tiny ImageNet)

%cd /content/Mase_EDGE
!wget http://cs231n.stanford.edu/tiny-imagenet-200.zip
!unzip tiny-imagenet-200.zip

In [ ]:
import optuna
import torch
import torch.nn as nn
import torchvision.models as models
import torch.nn.utils.prune as prune
import torch.export as exir
import torchvision.transforms as T
import torch.optim as optim
import subprocess
import json
import time
import re
import os, sys
from pathlib import Path
from typing import Any, Dict, Optional
from torchvision import datasets
# from mase.src.chop import MaseGraph
# import mase.src.chop.passes as passes

PROJECT_ROOT = os.path.dirname(os.path.abspath('__file__'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from EDGE_device.device_specifications import get_hardware_for_phone,load_phone_specs

### 1. Load Metrics from Android Phone

If you are trying to optimise and deploy the model on an Android phone, please plug in the phone to the laptop and make sure the developer mode is on. In settings make sure debugging mode is turned on. Run the following code cell to extract the hardware information. A JSON file will be created to hold the information and it will be used in the configuration setup.

In [ ]:
hw = get_hardware_for_phone() # Make sure the phone is connected and ADB is set up
print("Optimizing for:", hw["model"])
print(hw["gpu"], hw["cpu_cores"], hw["ram_gb"])

In [ ]:
# NPU/NNAPI hardware check (non-reference devices indicate accelerators like NPU)
# Multi-check for NPU/NNAPI hardware
checks = {
    "npu_prop": "shell getprop | grep -i npu",
    "ro.hardware": "shell getprop ro.hardware",
    "cpuinfo": "shell cat /proc/cpuinfo | head -20",
    "nn_hal": "shell ls /vendor/lib*/hw/ | grep neuralnetworks || echo none",
    "services_neural": "shell service list | grep -i neural"
}

results = {k: get_adb_output(v) for k, v in checks.items()}

# Heuristic NPU detection
npu_indicators = ["npu", "hexagon", "qcom", "kirin", "mali", "neuralnetworks"]
has_npu = any(any(ind in res.lower() for ind in npu_indicators) for res in results.values())
print(f"NPU/Accel likely: {'YES' if has_npu else 'NO'}")
print("Full results:", results)

In [ ]:
specs = load_phone_specs('EDGE_device/device_specifications.json')          # uses default PHONE_SPECS_FILE
print("Specs dict:", specs)
print("Keys:", list(specs.keys()))

if specs:
    hw = list(specs.values())[0]
    print(hw["model"], hw["gpu"], hw["cpu_cores"], hw["ram_gb"])
else:
    print("No specs found; JSON file missing or empty.")

### 2. Defining the Search Space

Here the code prepares the configurations for the pipeline

In [ ]:
def edge_optuna_config(trial):
    """Hardware-aware config: compression + mixed delegation."""

    base_opts = ["cpu"]
    if hw.get("prefers_gpu"):
        base_opts.extend(["xnnpack", "vulkan"])  # CPU/GPU
    if hw.get("has_npu"):
        base_opts.extend(["qnn", "nnapi"])       # NPU

    # Primary backend (where most ops go)
    primary_backend = trial.suggest_categorical("primary_backend", base_opts)

    # Whether to use mixed delegation at all
    use_mixed = trial.suggest_categorical("use_mixed_delegation", [False, True])

    # Compression 
    prune_ratio = trial.suggest_float("prune_ratio", 0.0, 0.7)
    quant_bits = trial.suggest_categorical("quant_bits", [8, 16])


    # Global default quant config
    if primary_backend in ["vulkan", "xnnpack"]:
        global_quant = trial.suggest_categorical("quant_config_global", ["int8", "fp16"])
    else:
        global_quant = "int8"

    # If not mixing, just return simple config
    if not use_mixed:
        return {
            "prune_ratio": prune_ratio,
            "quant_bits": quant_bits,
            "backend": primary_backend,
            "use_mixed_delegation": False,
            "delegation_plan": None,
            "quant_config_global": global_quant,
        }


    # Which units are allowed in the mix
    allowed_units = ["cpu"]
    if hw.get("prefers_gpu"):
        allowed_units.append("gpu")
    if hw.get("has_npu"):
        allowed_units.append("npu")

    # Choose a high-level “pattern” for splitting
    pattern = trial.suggest_categorical(
        "delegation_pattern",
        [
            "encoder_npu_decoder_cpu",     # early layers on NPU, later on CPU
            "early_layers_npu_late_cpu",  # similar but more flexible
            "attention_npu_rest_gpu", # attention on NPU, rest on GPU
            "gpu_primary_npu_offload", # give some to npu
            "cpu_primary_gpu_offload", # give some to gpu
        ],
    )

    # Per-unit ratios (0–1) – you can interpret them as fractions of ops or layers
    # and normalize in your partitioner to achieve load distribution. Optuna will search for the best balance.
    cpu_ratio = trial.suggest_float("cpu_ratio_raw", 0.0, 1.0)
    gpu_ratio = trial.suggest_float("gpu_ratio_raw", 0.0, 1.0) if "gpu" in allowed_units else 0.0
    npu_ratio = trial.suggest_float("npu_ratio_raw", 0.0, 1.0) if "npu" in allowed_units else 0.0

    # Will the partitioner auto round this in subgraph split?

    # Normalize
    total = cpu_ratio + gpu_ratio + npu_ratio
    if total == 0:
        cpu_ratio_norm, gpu_ratio_norm, npu_ratio_norm = 1.0, 0.0, 0.0
    else:
        cpu_ratio_norm = cpu_ratio / total
        gpu_ratio_norm = gpu_ratio / total
        npu_ratio_norm = npu_ratio / total

   
    # If consider NPU “high” more aggressive int8 there
    if hw.get("has_npu") and hw.get("npu_compute_level") == "high":
        npu_quant = trial.suggest_categorical("npu_quant_config", ["int8"])
    elif hw.get("has_npu"):
        npu_quant = trial.suggest_categorical("npu_quant_config", ["int8", "fp16"])
    else:
        npu_quant = None

    gpu_quant = None
    if hw.get("prefers_gpu"):
        gpu_quant = trial.suggest_categorical("gpu_quant_config", ["fp16", "int8"])

    cpu_quant = trial.suggest_categorical("cpu_quant_config", ["int8", "fp16"])

    delegation_plan = {
        "pattern": pattern,
        "ratios": {
            "cpu": cpu_ratio_norm,
            "gpu": gpu_ratio_norm,
            "npu": npu_ratio_norm,
        },
        "unit_quant": {
            "cpu": cpu_quant,
            "gpu": gpu_quant,
            "npu": npu_quant,
        },
    }

    return {
        "prune_ratio": prune_ratio,
        "quant_bits": quant_bits,
        "backend": primary_backend,
        "use_mixed_delegation": True,
        "delegation_plan": delegation_plan,
        "quant_config_global": global_quant,
    }

### 3. Model Optimisation

In [ ]:
# Inherit from Clarence's notebook and mase is used here

def edge_optimise_model(config: Dict[str, Any]) -> nn.Module:
    """ (pruning + PTQ) """
    
    device = "cuda" if torch.cuda.is_available() else "cpu"
    
    # Load + modify base model
    model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
    model.fc = nn.Linear(model.fc.in_features, 200)  # Tiny ImageNet
    model = model.to(device)
    
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=1e-4)
    
    transform = T.Compose([
        T.Resize(224), T.ToTensor(),
        T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])
    
    # 1 batch only (Optuna speed)
    train_dataset = datasets.ImageFolder(root="./tiny-imagenet-200/train", transform=transform)
    train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=16, shuffle=True)
    
    model.train()
    imgs, labels = next(iter(train_loader))
    imgs, labels = imgs.to(device), labels.to(device)
    optimizer.zero_grad()
    loss = criterion(model(imgs), labels)
    loss.backward()
    optimizer.step()
    
    example_inputs = torch.randn(1, 3, 224, 224).to(device)
    dummy_inputs = {"x": example_inputs}
    
    mg = MaseGraph(model)
    mg, _ = passes.init_metadata_analysis_pass(mg)
    mg, _ = passes.add_common_metadata_analysis_pass(mg, pass_args={"dummy_in": dummy_inputs})


    pruning_config = {
        "weight": {
            "sparsity": config["prune_ratio"],
            "method": "l1-norm",
            "scope": "local",
        },
        "activation": {
            "sparsity": config["prune_ratio"],
            "method": "l1-norm",
            "scope": "local",
        },
    }

    mg, _ = passes.prune_transform_pass(mg, pass_args=pruning_config)
    
    # Configurable quantization (Optuna param!)
    quantization_config = {
        "by": "type",
        "default": {"config": {"name": None}},
        "linear": {
            "config": {
                "name": "integer",
                "data_in_width": config["quant_bits"],
                "data_in_frac_width": config["quant_bits"] - 1,
                "weight_width": config["quant_bits"],
                "weight_frac_width": config["quant_bits"] - 1,
                "bias_width": 32,
                "bias_frac_width": 0,
            }
        },
    }
    
    mg, _ = passes.quantize_transform_pass(mg, pass_args=quantization_config)
    
    # Return FX GraphModule (ExecuTorch‑ready)
    ptq_model = torch.fx.GraphModule(mg.model, mg.fx_graph).cpu()
    return ptq_model



### 4. Export .pte file for Edge Device

In [ ]:
from pathlib import Path
from typing import Any, Dict

import torch
import torch.nn as nn

import executorch.exir as exir
from executorch.exir import to_edge

from executorch.exir.backend.backend_api import to_backend
from executorch.exir import ExecutorchProgramManager

# Import concrete partitioners
from executorch.backends.xnnpack.partition.xnnpack_partitioner import XnnpackPartitioner
from executorch.backends.vulkan.partition.vulkan_partitioner import VulkanPartitioner
from executorch.backends.qnn.partition.qnn_partitioner import QnnPartitioner
from executorch.backends.nnapi.partition.nnapi_partitioner import NnapiPartitioner
# For CPU-only you can just skip a backend, or define a trivial one if needed.


def edge_export_model(model: nn.Module, trial_id: int, config: Dict[str, Any]) -> str:
    """Export model to ExecuTorch .pte with backend partitioning."""

    model.eval()
    example_input = torch.randn(1, 3, 224, 224)

    exported = torch.export.export(model, (example_input,))
    edge: exir.EdgeProgramManager = to_edge(exported)

    # 3) Choose partitioner(s) based on config["backend"]
    backend = config["backend"]

    partitioners = []
    if backend == "xnnpack":
        partitioners.append(XnnpackPartitioner())
    elif backend == "vulkan":
        partitioners.append(VulkanPartitioner())
    elif backend == "qnn":
        partitioners.append(QnnPartitioner())
    elif backend == "nnapi":
        partitioners.append(NnapiPartitioner())
    elif backend == "cpu":
        # no delegation, leave as pure edge (no partitioner).
        partitioners = []
    else:
        # Fallback to XNNPACK
        partitioners.append(XnnpackPartitioner())

    # 4) Apply partitioners (if any)
    #    Note: you can chain multiple partitioners if you want mixed delegation.
    edge_prog = edge
    for part in partitioners:
        edge_prog = edge_prog.to_backend(part)

    # fusion would go here on edge_prog


    # Convert to ExecuTorch program and write .pte
    exec_prog: ExecutorchProgramManager = edge_prog.to_executorch()

    PTE_DIR = Path("pte_files")
    PTE_DIR.mkdir(exist_ok=True)
    pte_path = PTE_DIR / f"resnet18_trial_{trial_id}.pte"

    with open(pte_path, "wb") as f:
        exec_prog.write_to_file(f)

    return str(pte_path)

In [ ]:
def move_pte_to_app(pte_path: str, app_name: str):
    """Moves model to the app's private storage on the Android phone."""
    
    maybe_model_name = re.search(r'([^/\\]+\.pte)$', pte_path)
    model_name = maybe_model_name.group(1) if maybe_model_name else "model.pte"

    subprocess.run(["adb", "push", pte_path, "/data/local/tmp/"], check=True, timeout=30,)
    subprocess.run([   "adb", "shell", "run-as", app_name, "cp", f"/data/local/tmp/{model_name}", "files/model.pte",], check=True, timeout=30,)
    
def move_dataset_to_app(dataset_path: str, app_name: str):
    """Copy dataset into files/dataset/ so MV3Demo sees files/dataset/train|val|test/ (ImageFolder)."""
    dataset_name = os.path.basename(os.path.normpath(dataset_path))
    subprocess.run(["adb", "push", dataset_path, "/data/local/tmp/"], check=True, timeout=120)
    script = (
        "rm -rf files/dataset && mkdir -p files/dataset && "
        f"cp -R /data/local/tmp/{dataset_name}/. files/dataset/"
    )
    subprocess.run(
        ["adb", "shell", "run-as", app_name, "sh", "-c", script],
        check=True,
        timeout=120,
    )

### Benchmark for the trials on the phone

The functions push the optimised `.pte` and (once) a dataset folder into **Image Classification** (MV3Demo project) private storage (`files/model.pte`, `files/dataset/...`), then start **MainActivity** with action `com.image_classification_app.action.BENCHMARK`. **MaseOptimise** (in the app sources) runs ImageFolder eval and writes **`metrics_trial_<trial_id>.json`** into internal **`files/`** (same private storage as the model). The host polls with `adb exec-out run-as <app_id> cat files/metrics_trial_<id>.json` (requires a **debuggable** build so `run-as` works).

Pipeline: install the app once, set `EDGE_DEVICE_DATASET_PATH` (host folder with `train/`, `val/`, `test/`), set `EDGE_EVAL_SPLIT` if not `val`, then run the study.

Per trial: `adb push` .pte → `run-as` copy to `files/model.pte` → `am start` with extras `split`, `trial_id` → poll/pull metrics JSON.


In [ ]:
# Allow us to quantify the performance, and also act as the "oracle" for Optuna (returning the metrics)

# Android applicationId — must match app/build.gradle.kts defaultConfig.applicationId
IMAGE_CLASSIFICATION_APP_ID = "com.image_classification_app"

# Host path to a folder that contains train/, val/, and test/ subdirs (ImageFolder under each).
# Set before running the study; pushed to the phone only on trial 0 when not None.
EDGE_DEVICE_DATASET_PATH: Optional[str] = None  # e.g. "/path/to/MyDataset"
# Which split the app evaluates: "train" | "val" | "test"
EDGE_EVAL_SPLIT: str = "val"


def edge_benchmark_trial( trial_id: int, pte_path: str, dataset_path: Optional[str] = None,
    split: Optional[str] = None,app_name: str = IMAGE_CLASSIFICATION_APP_ID, ) -> Dict[str, float]:
    """Push → benchmark → metrics. MaseOptimise writes JSON to internal files/; host reads via run-as cat."""
    
    ds = dataset_path if dataset_path is not None else EDGE_DEVICE_DATASET_PATH
    ev_split = (split or EDGE_EVAL_SPLIT).lower()
    benchmark_action = f"{app_name}.action.BENCHMARK"
    try:
        move_pte_to_app(pte_path=pte_path, app_name=app_name)
        if trial_id == 0 and ds:
            move_dataset_to_app(dataset_path=ds, app_name=app_name)

        subprocess.run(
            ["adb", "shell", "am", "start", "-n", f"{app_name}/.MainActivity", "-a", benchmark_action,
                "-e", "split", ev_split, "-e", "trial_id", str(trial_id), ],
            check=True, timeout=30, )

        return edge_poll_metrics(trial_id, app_package=app_name)
    except Exception as e:
        print(f"Trial {trial_id} failed: {e}")
        return {"top1_acc": 0.0, "latency_p95_ms": 999.0, "memory_peak_mb": 0.0}


def edge_poll_metrics( trial_id: int, timeout: int = 120, app_package: str = IMAGE_CLASSIFICATION_APP_ID ) -> Dict[str, float]:
    """Poll metrics JSON from app internal filesDir via adb (debuggable builds: run-as cat)."""
    
    remote_rel = f"files/metrics_trial_{trial_id}.json"
    start = time.time()
    while time.time() - start < timeout:
        try:
            proc = subprocess.run(
                ["adb", "exec-out", "run-as", app_package, "cat", remote_rel],
                capture_output=True,
                timeout=10,
            )
            if proc.returncode == 0 and proc.stdout and proc.stdout.strip():
                data = json.loads(proc.stdout.decode("utf-8"))
                return {
                    "top1_acc": float(data.get("top1_acc", 0.0)),
                    "latency_p95_ms": float(data.get("latency_p95_ms", 999.0)),
                    "memory_peak_mb": float(data.get("memory_peak_mb", 0.0)),
                }
        except Exception:
            pass
        time.sleep(3)

    return {"top1_acc": 0.0, "latency_p95_ms": 999.0, "memory_peak_mb": 9999.0}


### 5. Defining the Objective Function

In [ ]:
def objective(trial):
    """Full pipeline: config, model, export, phone, score."""
    
    config = edge_optuna_config(trial)
    print(f"Trial {trial.number}: backend={config['backend']}, "
          f"prune={config['prune_ratio']:.1f}, quant={config['quant_config']}")
    
    # Optimize model
    model = edge_optimise_model(config)
    
    # Export for edge device
    pte_path = edge_export_model(model, trial.number, config)
    
    # Phone benchmark (where the pte file is run on the phone and metrics are collected)
    metrics = edge_benchmark_trial(
        trial.number, pte_path, split=EDGE_EVAL_SPLIT
    )
    
    # Multi‑objective score
    acc = metrics.get("top1_acc", 0.0)
    latency = metrics.get("latency_p95_ms", 999.0)
    memory = metrics.get("memory_peak_mb", 9999.0)
    
    # Score to maximise accuracy, but penalise latency and memory
    score = acc - 0.02 * (latency / 30.0) - 0.001 * (memory / 2000.0) # just an example need to review this
    
    trial.set_user_attr("top1_acc", acc)
    trial.set_user_attr("latency_ms", latency)
    trial.set_user_attr("memory_mb", memory)
    for k, v in config.items():
        trial.set_user_attr(k, v)
    
    print(f"  → acc={acc:.3f}, lat={latency:.1f}ms, score={score:.3f}")
    return score

### 6. Optuna Study

In [ ]:
# Create + run study
study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=42),
    study_name="resnet18_edge_opt",
    storage="sqlite:///optuna_resnet18.db",  # Save progress
    load_if_exists=True,
)

# Run 50 trials
study.optimize(objective, n_trials=50)

print(f"\nBest trial #{study.best_trial.number}")
print(f"   Score: {study.best_value:.3f}")
print("   Config:", study.best_params)

# Quick viz

optuna.visualization.plot_optimization_history(study).show()


### Extract the best model

In [ ]:
# Get best config
best_trial = study.best_trial
best_config = {k: v for k, v in best_trial.user_attrs.items() 
               if k in ["prune_ratio", "quant_bits", "backend", "fusion", "quant_config"]}

# Rebuild + export best model
best_model = edge_optimise_model(best_config)
final_pte = edge_export_model(best_model, 999, best_config)

print(f"Best model: {final_pte}")
print("Deploy with: adb push", final_pte, "/sdcard/resnet18_best.pte")
